In [19]:
from classes.style_encoder import StyleEncoder
from classes.mistral_7b import MistralGenerator
from helpers.encode_messages import encode_messages
from helpers.load_embeddings import load_embeddings
from helpers.build_prompt import build_prompt
from helpers.retrieve_top_k import retrieve_top_k
from transformers import CanineModel, CanineTokenizer
import pandas as pd
import torch
import faiss
import numpy as np
import pickle
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model_name = "style_retriever_author_contrastive"
# Load model
save_dir = "../models/" + model_name

tokenizer = CanineTokenizer.from_pretrained(save_dir)
encoder = CanineModel.from_pretrained(save_dir)

model = StyleEncoder()
model.encoder = encoder
model.proj.load_state_dict(torch.load(f"{save_dir}/projection_head.pt"))
model.eval()
model.to("cuda")

StyleEncoder(
  (encoder): CanineModel(
    (char_embeddings): CanineEmbeddings(
      (HashBucketCodepointEmbedder_0): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_1): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_2): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_3): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_4): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_5): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_6): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_7): Embedding(16384, 96)
      (char_position_embeddings): Embedding(16384, 768)
      (token_type_embeddings): Embedding(16, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (initial_char_encoder): CanineEncoder(
      (layer): ModuleList(
        (0): CanineLayer(
          (attention): CanineAttention(
            (self): CanineSelfAttention(
              (query): Linear(

In [ ]:
# load chat history and generate embeddings
df = pd.read_csv("../data/processed/private_full.csv")
df = df.dropna(subset=["Content"])
df["Content"] = df["Content"].astype(str)

print("Messages:", len(df))
print("Authors:", df["Author"].nunique())

texts = df["Content"].tolist()

embeddings = encode_messages(
    texts=texts,
    model=model,
    tokenizer=tokenizer,
    batch_size=8,
    device=device,
)

print("Embedding shape:", embeddings.shape)

records = [
    {
        "author": df.iloc[i]["Author"],
        "content": df.iloc[i]["Content"]
    }
    for i in range(len(df))
]

# Save to FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine similarity
index.add(embeddings.numpy())

print("Vectors in index:", index.ntotal)

output_dir = "../embeddings"
os.makedirs(output_dir, exist_ok=True)

faiss.write_index(index, os.path.join(output_dir, "style_index.faiss"))
with open(os.path.join(output_dir, "style_metadata.pkl"), "wb") as f:
    pickle.dump(records, f)
torch.save(embeddings, os.path.join(output_dir, "style_embeddings.pt"))

Messages: 40725
Authors: 2


100%|██████████| 5091/5091 [02:01<00:00, 41.91it/s]

Embedding shape: torch.Size([40725, 128])


In [12]:
index, records, embeddings = load_embeddings("../embeddings")

In [20]:
generator = MistralGenerator()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [21]:
# Example incoming message
incoming_message = "lol that was actually wild"

# Retrieve style examples
style_examples = retrieve_top_k(
    query_text=incoming_message,
    model=model,
    tokenizer=tokenizer,
    index=index,
    records=records,
    device=device,
    k=5
)

# Build prompt
prompt = build_prompt(incoming_message, style_examples)
print(prompt)
print("==================")

# Generate response suggestion

response = generator.generate(prompt)

print(response)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


You are generating a response suggestion.

Below are examples of how this person typically writes:

- but put him like really close
- loliday is under attack
- all those missed in rocket league
- all that is around 1500 words but you already finished in 5 minutes or so
- but until i think of one then it gonna be like that

Now write a response to the following message, matching the style above.

Message:
lol that was actually wild

Response:



You are generating a response suggestion.

Below are examples of how this person typically writes:

- but put him like really close
- loliday is under attack
- all those missed in rocket league
- all that is around 1500 words but you already finished in 5 minutes or so
- but until i think of one then it gonna be like that

Now write a response to the following message, matching the style above.

Message:
lol that was actually wild

Response:
but put your reaction right next to mine, it's getting crazy in here! loliday just got hacked and all those missed shots in Rocket League are haunting me. but I can't think of a good comeback right now, it's all that's been on my mind lately and it's around 1500 words but you managed to read
